# Data modeling reference

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-reference.ipynb)

A one-page summary of the data modeling section: the decisions, the vocabulary, and the SQL patterns. Each pattern
links to the lesson that explains it and runs against the shop's sample data, so you can change it and see the
effect. For practice, use the [Data modeling drills](data-modeling-drills.ipynb).

*Last reviewed: November 2026, against DuckDB 2.0.*

## Design checklist

1. Write the questions the model must answer.
2. Pick the business process; one process per fact table.
3. Declare the grain in one sentence; it is the fact's key.
4. Choose dimensions; give each attribute an SCD type (1 or 2, usually).
5. Choose measures; record additivity; put coarser measures on a coarser fact.
6. Look for traps: many-to-many (bridge), several dates (roles), levels (periodic snapshot), milestones
   (accumulating snapshot), late and early data.
7. Prove it with the SQL for one question.
8. Decide partitioning, clustering, load pattern and tests.

## Vocabulary

| Term | Meaning | Lesson |
|---|---|---|
| Grain | what one row of a table represents | [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) |
| Fact / dimension | measurements of events / descriptive context | [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) |
| Additive / semi-additive / non-additive | summable everywhere / not over time / never | [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) |
| Degenerate dimension | a dimension key on the fact with no dimension table (`order_id`) | [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) |
| Surrogate key | warehouse-assigned key per dimension row | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| Unknown member | reserved dimension row (`-1`) for unresolved keys | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| Role-playing dimension | one dimension joined in several roles (order date, ship date) | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| Junk dimension | combinations of leftover low-cardinality flags | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| Conformed dimension | one dimension shared by several facts | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| Bus matrix | processes × dimensions plan of a warehouse | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| Drill-across | aggregate each fact separately, then join on shared attributes | [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) |
| SCD Type 0 / 1 / 2 / 3 / 4 / 6 | keep / overwrite / version / previous column / mini-dimension / version + current column | [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb) |
| Point-in-time join | join a fact to the dimension version valid at its time | [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb) |
| Fan-out | a join that multiplies fact rows | [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb) |
| Chasm trap / fan trap | two child tables joined together / a parent filtered by a join to children | [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb) |
| Bridge table | resolves many-to-many, with weights for allocation | [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb) |
| Closure table | (ancestor, descendant) pairs for ragged hierarchies | [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb) |
| 1NF / 2NF / 3NF | one value per cell / whole key / nothing but the key | [Data modeling 5: normalized and operational models](data-modeling-05-normalized-and-operational-models.ipynb) |
| Hub / link / satellite | Data Vault business keys / relationships / historized attributes | [Data modeling 6: warehouse architectures](data-modeling-06-warehouse-architectures.ipynb) |
| Watermark | the point up to which a source has been loaded | [Data modeling 7: loading and late data](data-modeling-07-loading-and-late-data.ipynb) |
| Idempotent load | rerunning with the same input changes nothing | [Data modeling 7: loading and late data](data-modeling-07-loading-and-late-data.ipynb) |
| Inferred member | placeholder dimension row for an early-arriving fact | [Data modeling 7: loading and late data](data-modeling-07-loading-and-late-data.ipynb) |
| Partition pruning / zone map | skipping partitions / blocks by their values | [Data modeling 8: physical design and performance](data-modeling-08-physical-design-and-performance.ipynb) |
| Data contract | a published, checked schema and grain | [Data modeling 9: testing, metrics and change](data-modeling-09-testing-metrics-and-change.ipynb) |

## Decisions at a glance

| Situation | Choose |
|---|---|
| Events you count and sum | transaction fact |
| Levels over time (balances, stock, MRR) | periodic snapshot |
| A process with milestones | accumulating snapshot |
| An event or coverage with nothing to measure | factless fact |
| Measure at a coarser grain than the fact | a separate fact at that grain, or exact allocation |
| Attribute corrections | SCD Type 1 |
| Attribute whose history matters for reporting | SCD Type 2 |
| Need both "then" and "now" | SCD Type 6 |
| Fast-changing attributes | Type 4 mini-dimension |
| Many-to-many | bridge with weights |
| Ragged tree | parent-child table + closure table |
| Normalized dimension hierarchy | flatten into a star, unless large or shared |
| Querying layer | star schema; wide tables as marts where measured |
| Many changing sources, audit needs | Data Vault underneath the stars |
| Loads | `merge` on the key or delete-then-insert by partition |
| Large facts | partition by event date, cluster by common filters |

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_customer_changes` (the CRM's audit log of segment and city changes), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order), `src_payments` (one row per payment; an order can have several) and `src_promotions` and `src_order_promotions` (promotions applied to orders): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_customer_changes (customer_id int, changed_at timestamp, city varchar, segment varchar);
insert into src_customer_changes values  -- the CRM's audit log: one row each time a customer's details changed
    (101, '2025-11-02 00:00', 'Leeds', 'smb'),
    (102, '2025-11-20 00:00', 'York',  'smb'),
    (102, '2026-02-01 00:00', 'York',  'enterprise'),
    (103, '2025-12-05 00:00', 'Bath',  'enterprise'),
    (104, '2026-01-08 00:00', 'Hull',  'smb'),
    (104, '2026-02-15 00:00', 'Leeds', 'smb'),
    (105, '2026-01-19 00:00', 'Derby', 'smb'),
    (106, '2026-02-10 00:00', 'York',  'mid-market');
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_payments (payment_id varchar primary key, order_id int, paid_ts timestamp, method varchar, amount decimal(10, 2));
insert into src_payments values  -- order 1003 was paid in two parts; 1008 is unpaid and 1009 was cancelled
    ('P01', 1001, '2026-01-05 09:13', 'card',    76.99),
    ('P02', 1002, '2026-01-09 14:31', 'card',    60.00),
    ('P03', 1003, '2026-01-14 11:06', 'card',   132.50),
    ('P04', 1003, '2026-02-13 00:00', 'invoice', 100.00),
    ('P05', 1004, '2026-01-28 16:41', 'card',    34.99),
    ('P06', 1005, '2026-02-03 10:21', 'card',   114.99),
    ('P07', 1006, '2026-03-13 00:00', 'invoice', 400.00),
    ('P08', 1007, '2026-02-14 09:51', 'cash',    54.00),
    ('P09', 1010, '2026-02-27 08:31', 'cash',    33.00);
create table src_promotions (promo_code varchar primary key, promo_name varchar);
insert into src_promotions values ('WINTER10', 'Winter sale'), ('BULK', 'Bulk discount'), ('LOYAL', 'Loyalty reward');
create table src_order_promotions (order_id int, promo_code varchar, primary key (order_id, promo_code));
insert into src_order_promotions values (1002, 'WINTER10'), (1003, 'BULK'), (1003, 'LOYAL'), (1006, 'BULK');
""")
con.sql("show tables").show()

## SQL patterns

### Check a grain

The declared key must be unique; an empty result means it holds. ([lesson 1](data-modeling-01-grain-facts-and-dimensions.ipynb))

In [ ]:
con.sql("""
select order_id, line_no, count(*) as copies
from src_order_lines group by all having count(*) > 1
""").show()

### Surrogate keys with an unknown member

Facts resolve natural keys and fall back to `-1`. ([lesson 2](data-modeling-02-star-schema-design.ipynb))

In [ ]:
con.sql("""
create table dim_customer as
select -1 as customer_sk, null as customer_id, 'Unknown' as name
union all
select row_number() over (order by customer_id), customer_id, name from src_customers;

select o.order_id, coalesce(d.customer_sk, -1) as customer_sk
from (values (1001, 101), (2002, 999)) o(order_id, customer_id)
left join dim_customer d using (customer_id)
""").show()

### Date dimension

One row per day, with the business calendar precomputed. ([lesson 2](data-modeling-02-star-schema-design.ipynb))

In [ ]:
con.sql("""
select cast(strftime(d, '%Y%m%d') as int) as date_key, cast(d as date) as date, dayname(d) as weekday,
       isodow(d) >= 6 as is_weekend, case when month(d) >= 4 then year(d) + 1 else year(d) end as fiscal_year
from range(date '2026-03-30', date '2026-04-03', interval 1 day) t(d)
""").show()

### Drill across two facts

Aggregate each fact to the shared grain, then join. ([lesson 2](data-modeling-02-star-schema-design.ipynb))

In [ ]:
con.sql("""
with sales as (select o.customer_id, sum(l.qty * l.unit_price - l.discount) as sales
               from src_orders o join src_order_lines l using (order_id) where o.status <> 'cancelled' group by all),
     paid  as (select o.customer_id, sum(p.amount) as paid
               from src_payments p join src_orders o using (order_id) group by all)
select customer_id, sales, coalesce(paid, 0) as paid from sales left join paid using (customer_id) order by all
""").show()

### Type 2 dimension from a change log

Each change starts a version; the next change ends it. ([lesson 3](data-modeling-03-slowly-changing-dimensions.ipynb))

In [ ]:
con.sql("""
select customer_id, segment, changed_at as valid_from,
       coalesce(lead(changed_at) over w, timestamp '9999-12-31') as valid_to,
       lead(changed_at) over w is null as is_current
from src_customer_changes
where customer_id = 102
window w as (partition by customer_id order by changed_at)
""").show()

### Point-in-time join

Half-open range: `>= valid_from and < valid_to`, never `between`. ([lesson 3](data-modeling-03-slowly-changing-dimensions.ipynb))

In [ ]:
con.sql("""
with dim as (
    select customer_id, segment, changed_at as valid_from,
           coalesce(lead(changed_at) over (partition by customer_id order by changed_at), timestamp '9999-12-31') as valid_to
    from src_customer_changes
)
select o.order_id, o.order_ts, d.segment
from src_orders o join dim d
  on d.customer_id = o.customer_id and o.order_ts >= d.valid_from and o.order_ts < d.valid_to
where o.customer_id = 102
order by o.order_ts
""").show()

### Filter a parent by its children

`exists` keeps each parent once. ([lesson 4](data-modeling-04-relationships-and-double-counting.ipynb))

In [ ]:
con.sql("""
select count(*) as orders_with_coffee
from src_orders o
where exists (select 1 from src_order_lines l join src_products p using (sku)
              where l.order_id = o.order_id and p.category = 'Coffee')
""").show()

### Weighted bridge

Weights sum to 1 per parent, so allocations add up to the total. ([lesson 4](data-modeling-04-relationships-and-double-counting.ipynb))

In [ ]:
con.sql("""
with bridge as (select order_id, promo_code, 1.0 / count(*) over (partition by order_id) as weight from src_order_promotions),
revenue as (select order_id, sum(qty * unit_price - discount) as revenue from src_order_lines group by all)
select promo_code, sum(revenue) as full_credit, cast(sum(revenue * weight) as decimal(10, 2)) as allocated
from bridge join revenue using (order_id) group by all order by promo_code
""").show()

### Closure table for a tree

One recursive query; then any roll-up is a join. ([lesson 4](data-modeling-04-relationships-and-double-counting.ipynb))

In [ ]:
con.sql("""
with recursive tree(node, parent) as (
    values ('All', null), ('Grocery', 'All'), ('Hot drinks', 'Grocery'), ('Coffee', 'Hot drinks'), ('Kitchen', 'All')
),
closure(ancestor, descendant, depth) as (
    select node, node, 0 from tree
    union all
    select c.ancestor, t.node, c.depth + 1 from closure c join tree t on t.parent = c.descendant
)
select * from closure where ancestor = 'Grocery' order by depth
""").show()

### Latest event per key from CDC

Deduplicate a batch before merging it. ([lesson 7](data-modeling-07-loading-and-late-data.ipynb))

In [ ]:
con.sql("""
with cdc(lsn, order_id, status) as (values (1, 1008, 'shipped'), (2, 1008, 'delivered'), (2, 1008, 'delivered'), (3, 1011, 'placed'))
select * from cdc qualify row_number() over (partition by order_id order by lsn desc) = 1 order by order_id
""").show()

### Idempotent upsert

`merge` on the key: rerunning changes nothing. ([lesson 7](data-modeling-07-loading-and-late-data.ipynb))

In [ ]:
con.sql("""
create table fact_order_status as select order_id, status from src_orders where order_id < 1005;
merge into fact_order_status t using (select order_id, status from src_orders) s on t.order_id = s.order_id
when matched then update set status = s.status
when not matched then insert values (s.order_id, s.status);
select count(*) as orders from fact_order_status
""").show()

### Periodic snapshot from transactions

A spine of key × period, then a running sum. ([lesson 7](data-modeling-07-loading-and-late-data.ipynb))

In [ ]:
con.sql("""
with days as (select cast(d as date) as day from range(date '2026-01-13', date '2026-01-17', interval 1 day) t(d)),
billed as (select cast(order_ts as date) as day, sum(l.qty * l.unit_price - l.discount) as amount
           from src_orders o join src_order_lines l using (order_id) where o.customer_id = 103 group by all),
paid as (select cast(paid_ts as date) as day, sum(amount) as amount
         from src_payments p join src_orders o using (order_id) where o.customer_id = 103 group by all)
select d.day, sum(coalesce(b.amount, 0) - coalesce(p.amount, 0)) over (order by d.day) as balance
from days d left join billed b using (day) left join paid p using (day)
order by d.day
""").show()

## Model tests

| Test | Query returns the bad rows |
|---|---|
| unique grain | `select key from t group by key having count(*) > 1` |
| not null keys | `select * from fact where dim_sk is null` |
| relationships | `select f.* from fact f anti join dim d using (dim_sk)` |
| accepted values | `select distinct status from t where status not in (...)` |
| one current version | `select id from dim group by id having count(*) filter (where is_current) <> 1` |
| no overlapping versions | self-join on id where `a.valid_from < b.valid_to and b.valid_from < a.valid_to` |
| reconciliation | compare `sum(measure)` in the model with the source |

Fail the load when any returns rows ([Data modeling 9: testing, metrics and change](data-modeling-09-testing-metrics-and-change.ipynb)).